# Nombre de PR fusionnées par personne agrégées par semaine

Ce notebook récupère, via l'API GitHub, le nombre de *pull requests* (PR) fusionnées
pour **un ou plusieurs dépôts**, les regroupe par auteur et par semaine sur l'année écoulée,
puis affiche le résultat sous forme de graphique.

Les données récupérées sont **mises en cache** localement (un fichier CSV par dépôt).
Lors des exécutions suivantes, seules les PR plus récentes que la dernière date mise en cache
sont requêtées, ce qui réduit considérablement le nombre d'appels à l'API.

**Dépendances :** `requests`, `pandas`, `matplotlib`.

**Token GitHub :** l'API GitHub limite les appels non authentifiés à 60 requêtes par heure.
Pour lever cette limite, définissez la variable d'environnement `GITHUB_TOKEN`
avec un *Personal Access Token* (PAT) GitHub :

```bash
export GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxx
```

Sans token, le notebook fonctionne mais peut être limité sur de grands dépôts.

In [ ]:
import os
import datetime
import pathlib
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

## Paramètres

* `REPOS` — liste de dépôts à analyser sous la forme `[(owner, repo), ...]`.
* `CACHE_DIR` — répertoire où sont stockés les fichiers CSV de cache (un par dépôt).
  Utilisez `"."` pour enregistrer les fichiers à côté du notebook.

In [ ]:
REPOS = [
    ("sdpython", "teachpyx"),
    # ("sdpython", "onnx-extended"),  # ajoutez d'autres dépôts ici
]

# Répertoire de cache (créé automatiquement si nécessaire)
CACHE_DIR = pathlib.Path(".")

# Jeton d'authentification GitHub (optionnel mais recommandé)
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

## Récupération des PR fusionnées via l'API GitHub (avec cache)

Pour chaque dépôt :

1. On charge le fichier CSV de cache s'il existe (`prs_cache_{owner}_{repo}.csv`).
2. On détermine la date la plus récente déjà présente dans le cache.
3. On ne récupère auprès de l'API que les PR fusionnées **après** cette date
   (ou toutes si le cache est vide).
4. On fusionne les nouvelles PR avec le cache, on supprime les doublons
   et on élague les entrées datant de plus de 365 jours.
5. On sauvegarde le cache mis à jour sur disque.

In [ ]:
CACHE_DATE_FMT = "%Y-%m-%dT%H:%M:%S%z"


def _cache_path(cache_dir: pathlib.Path, owner: str, repo: str) -> pathlib.Path:
    """Retourne le chemin du fichier CSV de cache pour un dépôt."""
    safe = f"{owner}_{repo}".replace("/", "_")
    return cache_dir / f"prs_cache_{safe}.csv"


def load_cache(
    cache_dir: pathlib.Path, owner: str, repo: str
) -> pd.DataFrame:
    """Charge le cache CSV pour un dépôt (retourne un DataFrame vide si absent)."""
    path = _cache_path(cache_dir, owner, repo)
    if not path.exists():
        return pd.DataFrame(columns=["author", "merged_at", "repo"])
    df = pd.read_csv(path, parse_dates=["merged_at"])
    # S'assurer que la colonne est bien tz-aware (UTC)
    if df["merged_at"].dt.tz is None:
        df["merged_at"] = df["merged_at"].dt.tz_localize("UTC")
    else:
        df["merged_at"] = df["merged_at"].dt.tz_convert("UTC")
    return df


def save_cache(
    cache_dir: pathlib.Path, owner: str, repo: str, df: pd.DataFrame
) -> None:
    """Sauvegarde le DataFrame dans le fichier CSV de cache."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = _cache_path(cache_dir, owner, repo)
    df.to_csv(path, index=False, date_format=CACHE_DATE_FMT)


def fetch_merged_prs(
    owner: str,
    repo: str,
    token: str = "",
    fetch_since: datetime.datetime | None = None,
) -> list[dict]:
    """Récupère les PR fusionnées pour un dépôt à partir d'une date donnée.

    :param owner: propriétaire du dépôt GitHub
    :param repo: nom du dépôt GitHub
    :param token: jeton d'authentification GitHub (optionnel)
    :param fetch_since: si fourni, on s'arrête dès que ``merged_at`` est antérieur
        à cette date (les PR plus anciennes sont déjà en cache).
        Si ``None``, on remonte jusqu'à 365 jours en arrière.
    :return: liste de dictionnaires ``{author, merged_at, repo}``
    """
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    cutoff = fetch_since if fetch_since is not None else (
        datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=365)
    )

    results = []
    page = 1
    per_page = 100

    while True:
        url = (
            f"https://api.github.com/repos/{owner}/{repo}/pulls"
            f"?state=closed&per_page={per_page}&page={page}&sort=updated&direction=desc"
        )
        response = requests.get(url, headers=headers, timeout=30)
        try:
            response.raise_for_status()
        except requests.HTTPError as exc:
            status = exc.response.status_code
            if status == 401:
                raise RuntimeError(
                    "Authentification refusée (401). Vérifiez votre GITHUB_TOKEN."
                ) from exc
            if status == 403:
                raise RuntimeError(
                    "Accès refusé (403). Vous avez peut-être atteint la limite de l'API "
                    "GitHub (60 requêtes/h sans token). Définissez GITHUB_TOKEN."
                ) from exc
            if status == 404:
                raise RuntimeError(
                    f"Dépôt introuvable (404) : {owner}/{repo}. Vérifiez OWNER et REPO."
                ) from exc
            raise
        prs = response.json()

        if not prs:
            break

        stop = False
        for pr in prs:
            merged_at = pr.get("merged_at")
            if not merged_at:
                continue
            merged_dt = datetime.datetime.fromisoformat(merged_at.replace("Z", "+00:00"))
            if merged_dt <= cutoff:
                stop = True
                break
            author = (pr.get("user") or {}).get("login", "unknown")
            results.append({"author": author, "merged_at": merged_dt, "repo": f"{owner}/{repo}"})

        if stop:
            break

        page += 1

    return results


def load_prs_with_cache(
    owner: str, repo: str, token: str = "", cache_dir: pathlib.Path = pathlib.Path(".")
) -> pd.DataFrame:
    """Charge les PR fusionnées pour un dépôt en utilisant le cache local.

    * Si le cache existe, seules les PR plus récentes que la dernière entrée
      mise en cache sont récupérées via l'API.
    * Le cache est élagué pour ne conserver que les 365 derniers jours.
    * Le cache mis à jour est sauvegardé sur disque.

    :return: DataFrame avec les colonnes ``author``, ``merged_at``, ``repo``
    """
    now = datetime.datetime.now(datetime.timezone.utc)
    cutoff_365 = now - datetime.timedelta(days=365)

    cached_df = load_cache(cache_dir, owner, repo)

    if cached_df.empty:
        fetch_since = None  # récupérer toute l'année
        print(f"  {owner}/{repo} : cache vide, récupération complète…")
    else:
        # Relancer depuis le début de la journée du dernier enregistrement
        # pour ne pas manquer de PR fusionnées en cours de journée.
        latest = cached_df["merged_at"].max()
        fetch_since = latest.replace(hour=0, minute=0, second=0, microsecond=0)
        print(
            f"  {owner}/{repo} : cache chargé ({len(cached_df)} entrées), "
            f"récupération des PR depuis {fetch_since.date()}…"
        )

    new_prs = fetch_merged_prs(owner, repo, token, fetch_since=fetch_since)
    print(f"    → {len(new_prs)} nouvelle(s) PR(s) récupérée(s) via l'API.")

    if new_prs:
        new_df = pd.DataFrame(new_prs)
        combined = pd.concat([cached_df, new_df], ignore_index=True)
    else:
        combined = cached_df.copy()

    # Dédoublonnage et élagage
    combined.drop_duplicates(subset=["repo", "author", "merged_at"], inplace=True)
    combined = combined[combined["merged_at"] >= cutoff_365].copy()
    combined.sort_values("merged_at", inplace=True)
    combined.reset_index(drop=True, inplace=True)

    save_cache(cache_dir, owner, repo, combined)
    print(f"    → cache mis à jour ({len(combined)} entrées au total).")

    return combined


merged_prs_frames = []
for owner, repo in REPOS:
    repo_df = load_prs_with_cache(owner, repo, GITHUB_TOKEN, CACHE_DIR)
    merged_prs_frames.append(repo_df)

merged_prs = pd.concat(merged_prs_frames, ignore_index=True) if merged_prs_frames else pd.DataFrame()
print(f"\nTotal : {len(merged_prs)} PR(s) fusionnée(s) sur l'ensemble des dépôts.")

## Agrégation par auteur et par semaine

In [ ]:
df = merged_prs.copy() if not merged_prs.empty else pd.DataFrame(columns=["author", "merged_at", "repo"])

if df.empty:
    print("Aucune donnée à afficher.")
else:
    # Tronque la date au lundi de la semaine
    df["week"] = df["merged_at"].dt.to_period("W").dt.start_time

    weekly = (
        df.groupby(["repo", "author", "week"])
        .size()
        .reset_index(name="pr_count")
    )
    print(weekly.head(10))

## Tableau croisé (auteur × semaine, agrégé sur tous les dépôts)

In [ ]:
if not df.empty:
    # Agrégation sur tous les dépôts
    pivot = weekly.pivot_table(
        index="author", columns="week", values="pr_count", aggfunc="sum", fill_value=0
    )
    # Tri par nombre total de PR décroissant
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
    pivot

## Tableau croisé par dépôt

In [ ]:
if not df.empty and len(REPOS) > 1:
    for repo_name, grp in weekly.groupby("repo"):
        pvt = grp.pivot_table(
            index="author", columns="week", values="pr_count", aggfunc="sum", fill_value=0
        )
        pvt = pvt.loc[pvt.sum(axis=1).sort_values(ascending=False).index]
        print(f"\n=== {repo_name} ===")
        display(pvt)

## Visualisation : nombre de PR fusionnées par semaine (empilé par auteur)

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(14, 5))

    stacked_height = None
    weeks = pivot.columns  # DatetimeIndex
    week_nums = mdates.date2num(weeks.to_pydatetime())

    for author in pivot.index:
        values = pivot.loc[author].values
        if stacked_height is None:
            ax.bar(week_nums, values, width=5, label=author)
            stacked_height = values.copy()
        else:
            ax.bar(week_nums, values, width=5, bottom=stacked_height, label=author)
            stacked_height += values

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=4))
    plt.xticks(rotation=45, ha="right")
    ax.set_xlabel("Semaine")
    ax.set_ylabel("Nombre de PR fusionnées")
    repos_label = ", ".join(f"{o}/{r}" for o, r in REPOS)
    ax.set_title(f"PR fusionnées par semaine — {repos_label}")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1), title="Auteur")
    plt.tight_layout()
    plt.show()

## Visualisation : carte de chaleur (heatmap auteur × semaine)

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot) * 0.5)))

    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
    plt.colorbar(im, ax=ax, label="Nombre de PR")

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    # Affiche une étiquette de semaine sur 4
    step = max(1, len(pivot.columns) // 12)
    ax.set_xticks(range(0, len(pivot.columns), step))
    ax.set_xticklabels(
        [str(d)[:10] for d in pivot.columns[::step]], rotation=45, ha="right"
    )

    repos_label = ", ".join(f"{o}/{r}" for o, r in REPOS)
    ax.set_title(f"Heatmap des PR fusionnées — {repos_label}")
    ax.set_xlabel("Semaine")
    ax.set_ylabel("Auteur")
    plt.tight_layout()
    plt.show()